#  顧客留存分析｜舊客活化與再購行為分析

商業問題：  
顧客回購率與購買週期分析  
分析方法：  
- 計算會員總數與回購顧客數  
- 計算顧客回購率

In [17]:
SELECT COUNT(Customer_ID) AS [會員總人數],
       COUNT(case
             WHEN Total_Orders>1 THEN 1 
              END) AS [回購顧客數],
       ROUND(COUNT(case
             WHEN Total_Orders>1 THEN 1 
              END)*100.0/COUNT(Customer_ID),2) AS [回購率]
  FROM india_ecom.dbo.customers;

警告: 彙總或其他 SET 作業已刪除 Null 值。
(1 列受影響)

會員總人數 | 回購顧客數 | 回購率            
------+-------+----------------
40000 | 38406 | 96.020000000000
(1 個資料列)

總執行時間: 00:00:00.087

分析結果：  
會員總人數為 40,000 人，其中曾下單兩次以上的回購顧客為 38,406 人，回購率為 96.02% ，顯示多數會員具有二次以上購買行為。僅約 4% 為僅下單一次的顧客，整體回購表現相當穩定。

商業問題：  
顧客留存與流失分析  
分析方法：  
- 依顧客註冊月份建立 Cohort
- JOIN 訂單資料追蹤後續消費月份
- 統計各 Cohort 的活躍顧客數

In [18]:
WITH cohort_data AS (SELECT s.Customer_ID,
                            FORMAT(TRY_CONVERT(DATE,c.Registration_Date),'yyyy-MM') AS [註冊月份],
                            FORMAT(TRY_CONVERT(DATE,s.Order_Date),'yyyy-MM') AS [後續消費月份]
                       FROM india_ecom.dbo.customers AS c
                       JOIN india_ecom.dbo.sales AS s
                         ON c.Customer_ID=s.Customer_ID
                      WHERE s.Order_Status!='Cancelled')
SELECT [註冊月份],
       [後續消費月份],
       COUNT(DISTINCT Customer_ID) AS [活躍顧客數]
  FROM cohort_data
 GROUP BY [註冊月份],[後續消費月份]
 ORDER BY [註冊月份],[後續消費月份];

(300 個資料列受到影響)

註冊月份    | 後續消費月份  | 活躍顧客數
--------+---------+------
2023-06 | 2024-06 | 698  
2023-06 | 2024-07 | 722  
2023-06 | 2024-08 | 741  
2023-06 | 2024-09 | 698  
2023-06 | 2024-10 | 739  
2023-06 | 2024-11 | 693  
2023-06 | 2024-12 | 733  
2023-06 | 2025-01 | 703  
2023-06 | 2025-02 | 687  
2023-06 | 2025-03 | 746  
2023-06 | 2025-04 | 720  
2023-06 | 2025-05 | 716  
2023-06 | 2025-06 | 678  
2023-06 | 2025-07 | 753  
2023-06 | 2025-08 | 745  
2023-06 | 2025-09 | 693  
2023-06 | 2025-10 | 712  
2023-06 | 2025-11 | 742  
2023-06 | 2025-12 | 754  
2023-06 | 2026-01 | 699  
2023-06 | 2026-02 | 640  
2023-06 | 2026-03 | 760  
2023-06 | 2026-04 | 719  
2023-06 | 2026-05 | 704  
2023-06 | 2026-06 | 720  
2023-07 | 2024-06 | 708  
2023-07 | 2024-07 | 730  
2023-07 | 2024-08 | 714  
2023-07 | 2024-09 | 704  
2023-07 | 2024-10 | 741  
2023-07 | 2024-11 | 675  
2023-07 | 2024-12 | 766  
2023-07 | 2025-01 | 737  
2023-07 | 2025-02 | 668  
2023-07 | 2025-03 | 732  
2023-07 | 2025-04 | 71

分析結果：  
各註冊群組在後續消費月份的活躍顧客數大致落在 640～790 人之間波動，從最早的 2023-06 世代至後續月份，未見活躍顧客數隨時間拉長而持續下降的明顯趨勢。 2 月的活躍顧客數普遍較低，約落在 600～679 人，可能與當月天數較少有關，呈現季節性波動。

商業問題：  
商品評價與再購行為分析  
分析方法：  
- 計算顧客平均評價與訂單數
- 篩選訂單數達 2 筆以上的顧客

In [20]:
SELECT TOP(5000)Customer_ID, --因檔案過大，篩選前 5000 筆會員作為數據呈現，實際符合條件共 39431 人
       ROUND(AVG(Rating),2) AS [平均評價],
       COUNT(Order_ID) AS [訂單數]
  FROM india_ecom.dbo.sales
 GROUP BY Customer_ID
HAVING COUNT(Order_ID)>=2
 ORDER BY [平均評價] DESC,[訂單數] DESC;

警告: 彙總或其他 SET 作業已刪除 Null 值。
(5000 個資料列受到影響)

Customer_ID  | 平均評價 | 訂單數
-------------+------+----
CUST00034677 | 5    | 16 
CUST00024991 | 5    | 13 
CUST00033933 | 5    | 13 
CUST00032339 | 5    | 13 
CUST00033917 | 5    | 13 
CUST00023853 | 5    | 13 
CUST00034539 | 5    | 13 
CUST00017408 | 5    | 12 
CUST00034739 | 5    | 12 
CUST00008841 | 5    | 12 
CUST00020287 | 5    | 12 
CUST00018009 | 5    | 12 
CUST00009500 | 5    | 12 
CUST00037724 | 5    | 12 
CUST00027662 | 5    | 12 
CUST00014517 | 5    | 12 
CUST00036749 | 5    | 12 
CUST00022831 | 5    | 12 
CUST00031488 | 5    | 12 
CUST00037580 | 5    | 12 
CUST00008421 | 5    | 12 
CUST00010226 | 5    | 12 
CUST00031435 | 5    | 12 
CUST00000949 | 5    | 12 
CUST00021140 | 5    | 12 
CUST00018034 | 5    | 12 
CUST00029302 | 5    | 12 
CUST00029313 | 5    | 12 
CUST00014397 | 5    | 12 
CUST00003907 | 5    | 11 
CUST00010865 | 5    | 11 
CUST00024329 | 5    | 11 
CUST00012430 | 5    | 11 
CUST00013037 | 5    | 11 
CUST00021373 | 5   

分析結果：  
排序結果顯示，多位再購顧客的平均評價達 5 分，顯示此客群具有較高的平均評價。可進一步針對評價與訂單數皆高的回購顧客，評估會員忠誠方案或推薦計畫等經營方式。

商業問題：  
高價值顧客流失預警與挽回  
分析方法：  
- 計算顧客消費金額前 20% 門檻
- 篩選高消費且 180 天未下單的顧客
- 辨識高價值沉睡顧客名單

In [21]:
WITH spent AS (SELECT Customer_ID,
                      Total_Spent,
                      PERCENTILE_CONT(0.8) WITHIN GROUP(ORDER BY Total_Spent) OVER() AS [P80]
                 FROM india_ecom.dbo.customers),
last_order AS (SELECT Customer_ID,
       MAX(Order_Date) AS [最後訂單日]
  FROM india_ecom.dbo.sales
 WHERE Order_Status='Delivered'
  GROUP BY Customer_ID)

SELECT s.Customer_ID,
       s.Total_Spent,
       l.[最後訂單日]
  FROM spent AS s
  JOIN last_order AS l
    ON s.Customer_ID=l.Customer_ID
 WHERE Total_Spent>[P80]
   AND [最後訂單日]<DATEADD(DAY,-180,GETDATE());

(3466 個資料列受到影響)

Customer_ID  | Total_Spent   | 最後訂單日     
-------------+---------------+-----------
CUST00026203 | 197456.578125 | 2025-09-15
CUST00007301 | 286691.1875   | 2025-08-02
CUST00036873 | 219667.828125 | 2026-03-27
CUST00024841 | 255163.453125 | 2026-02-17
CUST00029673 | 511838.90625  | 2026-02-17
CUST00018910 | 228368.984375 | 2026-02-05
CUST00029414 | 321913.4375   | 2026-01-20
CUST00033577 | 194918.984375 | 2026-03-14
CUST00012132 | 249268.015625 | 2025-12-27
CUST00010689 | 342334.875    | 2026-03-09
CUST00007174 | 472697.5625   | 2026-01-08
CUST00036353 | 262757.09375  | 2025-11-07
CUST00002104 | 270855.21875  | 2026-03-25
CUST00008562 | 219612.75     | 2026-01-28
CUST00008982 | 217788.046875 | 2025-12-11
CUST00017278 | 204774.046875 | 2025-12-09
CUST00008593 | 202285.5      | 2025-09-11
CUST00000676 | 298968.625    | 2026-03-03
CUST00016125 | 244436.40625  | 2026-02-09
CUST00037970 | 223123.6875   | 2026-01-21
CUST00016783 | 276088.125    | 2026-02-22
CUST00002887 | 23

分析結果：  
累計消費金額位於前 20%（ P80 門檻為 19.4 萬），且超過 180 天未進行購買的顧客共 3,403 名。此客群具有較高歷史消費貢獻，但已長期未再次購買，可列為高價值沉睡顧客，進一步納入挽留與再行銷對象。